##**Code to Create the Stub**

In [1]:
%%writefile sat_pipeline.py
import asyncio
from dataclasses import dataclass, field
from typing import List

@dataclass
class VLMCommand:
    """Mock structure matching our Week 1 data models."""
    waypoints: List[List[float]]
    obstacles: List[str]
    confidence: float
    raw_response: str
    action: str = "MOVE"

async def mock_sat(image_path: str, api_key: str) -> VLMCommand:
    """
    Asynchronous stub representing the VLM image analysis endpoint.
    Simulates a 300ms processing delay and returns a mock command.
    """
    # Simulate the 300ms network round-trip / processing lag
    await asyncio.sleep(300 / 1000.0)

    # Return a structured mock result matching the production signature
    return VLMCommand(
        waypoints=[[0.25, -0.10, 0.45], [0.30, 0.00, 0.50]],
        obstacles=["left_safety_rail"],
        confidence=0.94,
        raw_response="Clear trajectory mapped. Obstacle detected on left safety rail."
    )

Writing sat_pipeline.py


In [2]:
import os
import shutil

# 1. Create the official production repository structure: src/hraf/
os.makedirs("hraf-middleware/src/hraf", exist_ok=True)

# 2. Move our stub file into the official source directory
shutil.copy("sat_pipeline.py", "hraf-middleware/src/hraf/sat_pipeline.py")

# 3. Create an empty __init__.py file to make the folder a recognized Python package
with open("hraf-middleware/src/hraf/__init__.py", "w") as f:
    pass

print("📁 Directory structure successfully established under hraf-middleware/src/hraf/")

📁 Directory structure successfully established under hraf-middleware/src/hraf/


In [3]:
import sys
# Inject our repository package source folder into Python's runtime search path
sys.path.append(os.path.abspath("hraf-middleware/src/hraf"))

# Test the import interface contract directly
try:
    from sat_pipeline import mock_sat
    print("✅ Success! The 'mock_sat' interface stub is cleanly imported into hraf-middleware.")
except ImportError as e:
    print(f"❌ Import failed: {e}")

✅ Success! The 'mock_sat' interface stub is cleanly imported into hraf-middleware.


In [4]:
%%writefile hraf-middleware/src/hraf/vlm_bridge.py
import asyncio
import sys
import os
from typing import Any

# Ensure our local directory is in the path for importing
sys.path.append(os.path.dirname(os.path.abspath(__file__)))
from sat_pipeline import mock_sat, VLMCommand

async def async_run_sat(image_path: str, api_key: str, timeout_seconds: float = 2.0) -> VLMCommand:
    """
    Wraps the VLM pipeline call inside a background thread worker (executor)
    and enforces a strict safety timeout.
    """
    # Get the active asynchronous running event loop
    loop = asyncio.get_running_loop()

    try:
        # Wrap the function execution with a strict max safety timeout window
        # loop.run_in_executor offloads the work to a background thread
        coroutine = loop.run_in_executor(
            None,
            lambda: asyncio.run(mock_sat(image_path, api_key))
        )

        # Await the execution but kill it if it takes longer than our timeout limit
        vlm_command = await asyncio.wait_for(coroutine, timeout=timeout_seconds)
        return vlm_command

    except asyncio.TimeoutError:
        print(f"⚠️ [TIMEOUT] VLM Pipeline exceeded safety limit of {timeout_seconds}s!")
        raise

Writing hraf-middleware/src/hraf/vlm_bridge.py


In [6]:
import asyncio
import time
import sys
import os

sys.path.append(os.path.abspath("hraf-middleware/src/hraf"))
from vlm_bridge import async_run_sat

async def run_concurrent_test_suite_v2():
    print("🚀 Running Updated Stage 3 Validation...")
    start_time = time.perf_counter()

    tasks = [
        async_run_sat(image_path=f"camera_frame_{i}.png", api_key="hraf_secure_key_v01")
        for i in range(10)
    ]

    results = await asyncio.gather(*tasks)
    total_duration_ms = (time.perf_counter() - start_time) * 1000.0

    print("\n=== CONCURRENT TEST METRICS ===")
    print(f"• Successfully gathered: {len(results)} VLMCommand outputs.")
    print(f"• Total pipeline execution time: {total_duration_ms:.2f} ms")
    print("===============================\n")

    # 10 sequential calls would take >3000ms. Anything well below 1000ms proves concurrency!
    if total_duration_ms < 1000.0:
        print("✅ SUCCESS: Verification passed! Total time is well under 3,000ms, proving all 10 requests ran simultaneously without blocking.")
    else:
        print("❌ FAILURE: Pipeline execution took too long.")

await run_concurrent_test_suite_v2()

🚀 Running Updated Stage 3 Validation...

=== CONCURRENT TEST METRICS ===
• Successfully gathered: 10 VLMCommand outputs.
• Total pipeline execution time: 605.86 ms

✅ SUCCESS: Verification passed! Total time is well under 3,000ms, proving all 10 requests ran simultaneously without blocking.
